# Handling Data & Concept Drift

A model trained today assumes tomorrow's data looks like today's. In the real world it rarely does: user behaviour moves, sensors age, an upstream pipeline changes units, a pandemic rewrites demand. When the world moves and the *frozen* model does not, accuracy quietly rots. This notebook builds a small **monitoring system** that watches a live stream and raises an alarm the moment the model starts to degrade.

We separate two very different failure modes:

| Kind | What changes | Formal statement | Does the old label rule still hold? |
|------|--------------|------------------|-------------------------------------|
| **Covariate / data drift** | the *inputs* | $P(X)$ shifts, $P(y\mid X)$ fixed | yes |
| **Concept drift** | the *input→label relationship* | $P(y\mid X)$ shifts | **no** |

The joint distribution factorises as $P(X, y) = P(X)\,P(y\mid X)$. **Covariate drift** moves the first factor (the feature cloud slides to a new place), while the rule mapping features to labels is untouched. **Concept drift** moves the second factor (the same feature vector now deserves a different label — the decision boundary itself has rotated/flipped).

This distinction is the whole point of the notebook, because it dictates *how you can even detect the problem*:

- Covariate drift is visible from the **inputs alone** — you can spot it with **no labels at all** by comparing the incoming feature distribution to a reference window (KS test, PSI). This is label-free and instant.
- Concept drift is (often) **invisible in the inputs** — $P(X)$ can look identical while the answer key has changed. To see it you must compare predictions to **ground-truth labels**, which in production usually arrive with a **delay** (you learn whether a loan defaulted months later).

We simulate a time-ordered stream with a covariate shift at a *known* index and a concept shift at another, then validate that our detectors fire at the right moments.

In [ ]:
import numpy as np                                  # arrays + the RNG that drives the whole stream
import pandas as pd                                 # tidy per-batch monitoring log
import matplotlib.pyplot as plt                     # plotting only
import seaborn as sns                               # nicer default styling for the scatter/plots
from scipy.stats import ks_2samp                    # two-sample Kolmogorov–Smirnov test (input-drift stat)
from sklearn.ensemble import RandomForestClassifier # the model we freeze and monitor
from sklearn.metrics import accuracy_score, roc_auc_score  # the two labelled performance metrics

# --- Seed EVERYTHING so the notebook is bit-for-bit reproducible run to run ---
SEED = 42
rng = np.random.default_rng(SEED)  # single Generator -> every batch is drawn from this one stream
np.random.seed(SEED)               # legacy global RNG, belt-and-suspenders for any library that uses it
sns.set_theme(style="whitegrid")   # apply seaborn styling to all matplotlib figures below

print("environment ready")

## 1. A synthetic, time-ordered stream

We emit `N_BATCHES` batches in order (think: one batch per day). Each row is a 2-D feature vector $x = (x_0, x_1)$. The **stable ground-truth rule** is a parabola — a point is class 1 when it sits *above* the curve:

$$g(x) = x_1 - \left(\tfrac{1}{2}x_0^2 - 1\right), \qquad y = \mathbf{1}\big[\,s\cdot g(x) > 0\,\big]$$

where $s \in \{+1, -1\}$ is a **rule sign**. A few percent of labels are randomly flipped so the problem is not trivially separable (irreducible noise, like the real world).

The stream has three regimes, with the drift happening at **known indices** so we can score the detector:

| Batches | $P(X)$ (feature mean) | rule sign $s$ | Regime |
|---------|-----------------------|---------------|--------|
| `0 – 9`   | $\mathcal{N}([0,0],\,\sigma)$   | $+1$ | **stable** (reference) |
| `10 – 19` | $\mathcal{N}([3,4],\,\sigma)$   | $+1$ | **covariate drift** — the cloud slides; *the rule is untouched* |
| `20 – 29` | $\mathcal{N}([0,0],\,\sigma)$   | $-1$ | **concept drift** — cloud back to normal; *the boundary flips* |

Notice the deliberate design: at the concept-drift point the feature distribution returns **exactly** to the reference — so an input-only monitor will see *nothing*, yet the labels have inverted. That is the trap concept drift sets, and why label-based monitoring is indispensable.

In [ ]:
# --- Known configuration of the stream (these are the 'ground truth' we validate against) ---
N_BATCHES   = 30     # total batches in the stream
BATCH_SIZE  = 200    # rows per batch
TRAIN_END   = 5      # batches 0..4 are used to TRAIN the frozen model
REF_END     = 10     # batches 0..9 form the stable REFERENCE window for drift stats
COV_DRIFT   = 10     # TRUE covariate-drift onset (P(X) shifts here)
CONCEPT_DRIFT = 20   # TRUE concept-drift onset (P(y|X) flips here)
SIGMA       = 1.5    # per-feature standard deviation of every Gaussian blob
NOISE       = 0.05   # fraction of labels randomly flipped (irreducible error)


def make_batch(n, mean, rule_sign, rng, noise=NOISE):
    """Draw one batch: features from a Gaussian at `mean`, labels from the (possibly flipped) parabola rule."""
    X = rng.normal(loc=mean, scale=SIGMA, size=(n, 2))   # P(X): a 2-D Gaussian cloud centred at `mean`
    g = X[:, 1] - (0.5 * X[:, 0] ** 2 - 1.0)             # signed distance to the parabola boundary
    y = (rule_sign * g > 0).astype(int)                  # P(y|X): rule_sign=+1 normal, -1 = boundary flipped
    flip = rng.random(n) < noise                         # pick ~5% of rows...
    y[flip] = 1 - y[flip]                                 # ...and flip their labels -> label noise
    return X, y.astype(int)


# Build the full stream as a list of (X, y) tuples, one per batch, IN TIME ORDER.
stream = []
for t in range(N_BATCHES):
    if t < COV_DRIFT:            # regime 1: stable
        mean, sign = [0.0, 0.0], +1
    elif t < CONCEPT_DRIFT:      # regime 2: covariate drift -> cloud moves, rule unchanged
        mean, sign = [3.0, 4.0], +1
    else:                        # regime 3: concept drift -> cloud back home, rule sign flipped
        mean, sign = [0.0, 0.0], -1
    stream.append(make_batch(BATCH_SIZE, mean, sign, rng))

print(f"built {len(stream)} batches of {BATCH_SIZE} rows each")
print(f"true covariate-drift onset : batch {COV_DRIFT}")
print(f"true concept-drift onset   : batch {CONCEPT_DRIFT}")

### What each regime looks like

One representative batch from each regime. Watch two things: **where the cloud sits** (that is $P(X)$) and **which colour lives where** (that is $P(y\mid X)$).

In [ ]:
# Pick one batch from each regime to illustrate the two kinds of movement.
examples = [(2, "Stable (batch 2)"), (14, "Covariate drift (batch 14)"), (24, "Concept drift (batch 24)")]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharex=True, sharey=True)
for ax, (t, title) in zip(axes, examples):
    X, y = stream[t]
    # colour by class: same colormap across panels so colours mean the same thing everywhere
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", s=14, edgecolor="k", linewidth=0.2)
    # overlay the STABLE parabola boundary (x1 = 0.5*x0^2 - 1) as a reference in every panel
    xs = np.linspace(-4, 7, 200)
    ax.plot(xs, 0.5 * xs ** 2 - 1.0, "k--", lw=1, label="stable boundary")
    ax.set_title(title); ax.set_xlabel("x0"); ax.set_ylabel("x1"); ax.legend(loc="upper left", fontsize=8)
axes[0].set_xlim(-5, 8); axes[0].set_ylim(-6, 10)  # shared limits so the CLOUD MOVEMENT is visible
plt.tight_layout(); plt.show()

# Read the panels: stable & covariate share the SAME rule (colours agree with the dashed curve), but
# the covariate cloud has slid up-right. The concept panel sits back on the reference cloud, yet the
# colours are INVERTED relative to the dashed boundary -> the rule, not the inputs, changed.

## 2. Train a model on the stable window, then freeze it

This mimics production: you fit once on historical data, ship it, and it stops learning. We train a `RandomForestClassifier` on batches `0–4` and **never refit it** (until the very end, when we demonstrate recovery). A tree ensemble is a good subject for a drift demo because it does **not extrapolate** — when the inputs slide into regions it never saw during training, its predictions get shaky, which is exactly the covariate-drift symptom we want to observe.

In [ ]:
# Stack the training batches (0..TRAIN_END-1) into one design matrix + label vector.
X_train = np.vstack([stream[t][0] for t in range(TRAIN_END)])
y_train = np.concatenate([stream[t][1] for t in range(TRAIN_END)])

# Fit the model once. random_state makes the forest itself reproducible.
model = RandomForestClassifier(n_estimators=100, random_state=SEED)
model.fit(X_train, y_train)  # after this line the model is CONCEPTUALLY FROZEN

# Build the REFERENCE feature window (all stable batches 0..REF_END-1). Every drift statistic below
# compares an incoming batch against THIS fixed reference — it is our definition of 'normal'.
X_ref = np.vstack([stream[t][0] for t in range(REF_END)])

print(f"trained on {len(y_train)} rows (batches 0..{TRAIN_END-1}), class balance = {y_train.mean():.2f}")
print(f"reference window for drift stats = batches 0..{REF_END-1} ({len(X_ref)} rows)")

## 3. Two families of monitor

**(a) Labelled performance monitors — need $y$.** For each incoming batch we compute rolling **accuracy** and **ROC–AUC** against the true labels. These directly answer "is the model still right?" but they are only available once labels arrive, which in production is often **delayed** (sometimes by weeks). AUC is worth watching alongside accuracy because it tracks the *ranking* quality and can fall even when accuracy is propped up by a shifting class balance.

**(b) Input-distribution monitors — need only $X$.** These compare the incoming features to the reference window and need **no labels**, so they run instantly on live traffic.

*Kolmogorov–Smirnov (KS) two-sample statistic* — the largest gap between the two empirical CDFs of a feature:

$$D = \sup_x \left| F_{\text{ref}}(x) - F_{\text{cur}}(x) \right| \in [0, 1]$$

*Population Stability Index (PSI)* — bin the reference into deciles, then measure how much probability mass moved between bins:

$$\text{PSI} = \sum_{b=1}^{B} \left(a_b - e_b\right)\,\ln\!\frac{a_b}{e_b}$$

where $e_b$ / $a_b$ are the expected (reference) / actual (current) fractions in bin $b$. Rule of thumb: PSI $< 0.1$ no shift, $0.1–0.25$ moderate, $> 0.25$ major shift. We report the **worst feature** for each statistic.

In [ ]:
def psi(reference, current, bins=10, eps=1e-4):
    """Population Stability Index between a reference and current 1-D sample (hand-rolled)."""
    # Cut points = deciles of the REFERENCE. Force the outer edges to +/-inf so any out-of-range
    # value in `current` still lands in the first/last bin instead of being dropped.
    edges = np.quantile(reference, np.linspace(0, 1, bins + 1))
    edges[0], edges[-1] = -np.inf, np.inf
    e = np.histogram(reference, edges)[0] / len(reference)  # expected fraction per bin
    a = np.histogram(current,   edges)[0] / len(current)    # actual  fraction per bin
    e = np.clip(e, eps, None)                               # clip zeros so ln() and division stay finite
    a = np.clip(a, eps, None)
    return float(np.sum((a - e) * np.log(a / e)))           # sum of per-bin population shift


def input_drift_stats(X_reference, X_current):
    """Worst-feature KS statistic and worst-feature PSI — both computed from features ONLY (no labels)."""
    ks  = max(ks_2samp(X_reference[:, j], X_current[:, j]).statistic for j in range(X_reference.shape[1]))
    ps  = max(psi(X_reference[:, j], X_current[:, j])               for j in range(X_reference.shape[1]))
    return ks, ps


# Quick sanity check on a stable batch vs a covariate-drifted batch.
print("stable batch 2   :", tuple(round(v, 3) for v in input_drift_stats(X_ref, stream[2][0])))
print("covariate batch 14:", tuple(round(v, 3) for v in input_drift_stats(X_ref, stream[14][0])))

## 4. The drift detector

We run **two independent detectors** and, critically, note *which* one catches *which* drift:

1. **Input-drift detector** (label-free): alarm when worst-feature $\text{KS} > 0.2$ **or** worst-feature $\text{PSI} > 0.25$.
2. **Performance detector** (needs labels): alarm when rolling $\text{AUC} < 0.8$.

A single batch can trip a false alarm, so a robust production rule would require *k* consecutive breaches; here the signals are strong and clean enough that one batch suffices. We stream every batch through the frozen model, log all metrics, and record the **first** batch (from the covariate onset onward) at which each detector fires — then compare to the known drift indices.

In [ ]:
# --- Alarm thresholds ---
KS_TH, PSI_TH, AUC_TH = 0.20, 0.25, 0.80

rows = []                      # one dict per batch -> becomes the monitoring DataFrame
input_alarm = None             # first batch the label-FREE detector fires
perf_alarm  = None             # first batch the label-BASED detector fires

for t in range(N_BATCHES):
    X, y = stream[t]
    proba = model.predict_proba(X)[:, 1]          # frozen model's P(y=1) for this batch
    pred  = (proba >= 0.5).astype(int)            # hard 0/1 prediction

    acc = accuracy_score(y, pred)                 # labelled metric 1: accuracy
    auc = roc_auc_score(y, proba)                 # labelled metric 2: ranking quality
    ks, ps = input_drift_stats(X_ref, X)          # label-free input-distribution stats

    input_fire = (ks > KS_TH) or (ps > PSI_TH)    # did the input detector trip on this batch?
    perf_fire  = auc < AUC_TH                      # did the performance detector trip?

    # Record the FIRST firing at/after the covariate onset (batches 0..9 are the reference baseline).
    if input_fire and input_alarm is None and t >= COV_DRIFT:
        input_alarm = t
    if perf_fire and perf_alarm is None and t >= COV_DRIFT:
        perf_alarm = t

    rows.append(dict(batch=t, accuracy=acc, auc=auc, ks=ks, psi=ps,
                     input_fire=input_fire, perf_fire=perf_fire))

log = pd.DataFrame(rows)      # tidy monitoring log, one row per batch
print(log.round(3).to_string(index=False))

In [ ]:
# --- Compare each detector's first alarm to the TRUE drift indices ---
print("TRUE covariate-drift onset :", COV_DRIFT)
print("TRUE concept-drift onset   :", CONCEPT_DRIFT)
print()
print(f"INPUT detector (KS/PSI, no labels) first fired at batch {input_alarm} "
      f"-> lag {None if input_alarm is None else input_alarm - COV_DRIFT} batch(es) after covariate drift")
print(f"PERF  detector (AUC, needs labels) first fired at batch {perf_alarm} "
      f"-> lag {None if perf_alarm is None else perf_alarm - CONCEPT_DRIFT} batch(es) after concept drift")
print()
# The key teaching result: the label-FREE monitor caught the COVARIATE shift, but stayed SILENT through
# the concept shift (features looked normal). Only the label-BASED monitor caught the CONCEPT shift.
print("input detector fired during concept regime? ",
      bool(log.loc[log.batch >= CONCEPT_DRIFT, 'input_fire'].any()))
print("perf  detector fired during concept regime? ",
      bool(log.loc[log.batch >= CONCEPT_DRIFT, 'perf_fire'].any()))

## 5. Visualising the monitors over time

Two views. **Top:** rolling accuracy and AUC, with the true drift onsets (dashed) and the detector alarms (dotted) marked. **Bottom:** the label-free input-drift statistics (KS, PSI) against their thresholds. Read them together to see the division of labour — the input stat spikes at the covariate shift, while the performance curve is the *only* thing that reacts to the concept shift.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 8), sharex=True)

# ---- Top panel: labelled performance over time ----
ax1.plot(log.batch, log.accuracy, marker="o", ms=4, label="rolling accuracy")
ax1.plot(log.batch, log.auc,      marker="s", ms=4, label="rolling AUC")
ax1.axhline(AUC_TH, color="grey", ls=":", label=f"AUC threshold {AUC_TH}")
ax1.axvline(COV_DRIFT,     color="orange", ls="--", lw=2, label="TRUE covariate drift")
ax1.axvline(CONCEPT_DRIFT, color="red",    ls="--", lw=2, label="TRUE concept drift")
if perf_alarm is not None:   # dotted vertical line where the performance detector alarmed
    ax1.axvline(perf_alarm, color="purple", ls=":", lw=2, label=f"PERF alarm (batch {perf_alarm})")
ax1.set_ylabel("score"); ax1.set_ylim(0, 1.05)
ax1.set_title("Labelled performance (needs ground-truth y)"); ax1.legend(loc="center left", fontsize=8)

# ---- Bottom panel: label-free input-drift statistics over time ----
ax2.plot(log.batch, log.ks,  marker="o", ms=4, color="teal",    label="KS statistic (worst feature)")
ax2.axhline(KS_TH, color="teal", ls=":", label=f"KS threshold {KS_TH}")
ax2.plot(log.batch, log.psi, marker="^", ms=4, color="darkgreen", label="PSI (worst feature)")
ax2.axhline(PSI_TH, color="darkgreen", ls=":", label=f"PSI threshold {PSI_TH}")
ax2.axvline(COV_DRIFT,     color="orange", ls="--", lw=2, label="TRUE covariate drift")
ax2.axvline(CONCEPT_DRIFT, color="red",    ls="--", lw=2, label="TRUE concept drift")
if input_alarm is not None:  # dotted vertical line where the input detector alarmed
    ax2.axvline(input_alarm, color="purple", ls=":", lw=2, label=f"INPUT alarm (batch {input_alarm})")
# PSI can spike to large values under a big shift; log-scale keeps both KS (~0–1) and PSI readable.
ax2.set_yscale("symlog", linthresh=0.1)
ax2.set_xlabel("batch (time)"); ax2.set_ylabel("drift statistic (symlog)")
ax2.set_title("Input-distribution drift (label-free)"); ax2.legend(loc="center left", fontsize=8)

plt.tight_layout(); plt.show()

## 6. Responding to drift: retrain on recent data

Detection is only half the job — you have to *act*. The standard responses are:

- **Retrain** on fresh, post-drift data (what we do below).
- **Windowing** — always train on a sliding window of the most recent batches so the model tracks a moving world automatically.
- **Alerting / human-in-the-loop** — page an on-call owner, or fall back to a safe default while a human investigates.

Here, once the performance detector flags concept drift, we retrain the forest on the two most recent batches (the *new* regime) and replay the remaining stream to watch accuracy recover.

In [ ]:
# Trigger retraining off the performance alarm (fall back to the known onset if it somehow didn't fire).
retrain_at = perf_alarm if perf_alarm is not None else CONCEPT_DRIFT

# Fit a fresh forest on the two batches immediately available after the alarm (the post-drift regime).
X_new = np.vstack([stream[retrain_at][0], stream[retrain_at + 1][0]])
y_new = np.concatenate([stream[retrain_at][1], stream[retrain_at + 1][1]])
model_retrained = RandomForestClassifier(n_estimators=100, random_state=SEED).fit(X_new, y_new)

# Replay every subsequent batch through BOTH the frozen and the retrained model to compare accuracy.
later = list(range(retrain_at + 2, N_BATCHES))
acc_frozen    = [accuracy_score(stream[t][1], model.predict(stream[t][0]))            for t in later]
acc_retrained = [accuracy_score(stream[t][1], model_retrained.predict(stream[t][0]))  for t in later]

print(f"retrained on batches {retrain_at} & {retrain_at + 1}")
print(f"mean accuracy on batches {later[0]}..{later[-1]}:")
print(f"   frozen model    = {np.mean(acc_frozen):.3f}  (still using the stale concept)")
print(f"   retrained model = {np.mean(acc_retrained):.3f}  (recovered)")

# Plot the recovery: frozen model stays on the floor, retrained model climbs back to ~0.9.
plt.figure(figsize=(9, 4))
plt.plot(later, acc_frozen,    marker="o", label="frozen model (no action)")
plt.plot(later, acc_retrained, marker="s", label="retrained after alarm")
plt.axhline(0.5, color="grey", ls=":", label="chance")
plt.ylim(0, 1.05); plt.xlabel("batch (time)"); plt.ylabel("accuracy")
plt.title("Recovery after retraining on post-drift data"); plt.legend(); plt.tight_layout(); plt.show()

## 7. Takeaways

**Covariate vs concept drift.** Covariate drift moves $P(X)$ (the inputs land somewhere new) while the input→label rule holds; concept drift moves $P(y\mid X)$ (the same input now means something different). Our stream showed both at known times, and the detectors fired essentially *on* those times.

**Why input monitoring is not enough — and why labels matter.** The KS/PSI monitor is wonderful because it needs **no labels**: it caught the covariate shift instantly and for free. But it was completely **blind** to the concept shift — by construction the features had returned to normal, so $P(X)$ looked healthy while the answer key had inverted. Only the **labelled** performance monitor (accuracy/AUC) could see that. The catch: ground-truth labels in production usually arrive **late**, so concept drift is intrinsically harder and slower to detect. In practice you run **both** — cheap label-free input monitors for early warning, and label-based performance monitors (as labels trickle in) for the drift that hides in the inputs.

**Common responses.** *Retrain* on recent data (shown here — accuracy recovered from the floor back to ~0.9); use a *sliding window* so the model continuously tracks a moving distribution; and wire up *alerting* so a human is looped in before a silent failure becomes an incident. Robustify detectors by requiring several consecutive breaches to avoid false alarms, and always keep a fixed reference window that encodes what 'normal' means.